# Setup

# Network Build — Synapse Edge Lists

**Purpose:** Builds synapse edge lists from eCREST reconstruction files. For each reconstructed
cell, reads post-synaptic and pre-synaptic annotations from `.json` files and outputs a flat
synapse dataframe.

**Inputs:**
- `EM_data_published/reconstructions_published/` — eCREST `.json` reconstruction files

**Outputs:**
- `EM_data_published/data_processed_published/df_postsyn.csv`
- `EM_data_published/data_processed_published/df_presyn.csv`

**Used by:** `Analyses_published.ipynb` (Sensory Input, Granule Cell Axons, MG Connectivity sections)

**Launch requirement:** Run `jupyter lab` from `efish_em/Notebooks_Jupyter/`

In [ ]:
from pathlib import Path
import sys
import numpy as np
from numpy import array
import pandas as pd
from tqdm import tqdm

DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))
import AnalysisCode as efish

DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'


In [ ]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'


# Get all cells info

In [72]:
nodefiles = efish.get_cell_filepaths(dirpath)

## Base Segments

In [74]:
base_segments = {}
for x,f in nodefiles.items():
    cell_data = efish.load_ecrest_celldata(f) 
    base_segments[cell_data['metadata']['main_seg']['base']] = cell_data['base_segments']
    
    try:
        assert cell_data['metadata']['main_seg']['base'] == x
    except:
        print(x,cell_data['metadata']['main_seg']['base'])

# Build Graph

In [82]:
synanno_type = 'post-synaptic' # switch to 'pre-synaptic' for presynaptic inputs to the reconstructed cells
vx_sizes = [16, 16, 30]

## find edges and set the cell-structure attribute of the edge based on which part of the cell the edge goes to
edge_list = []

with tqdm(total=len(nodefiles.keys())) as pbar:
    for x_pre,f_pre in nodefiles.items():
        pbar.update(1)
        pre = efish.load_ecrest_celldata(f_pre)
        
        # if pre['metadata']['cell-type']['manual'] in network_types: 
        this_type_annotations=[]
        if synanno_type in pre['end_points'].keys():
            for pos, point in enumerate(pre['end_points'][synanno_type]):
                if point[-1] not in ['annotatePoint','annotateBoundingBox','annotateSphere']:
                    point = (point, 'annotatePoint') 
                this_type_annotations.append(point)
            # rewrite end_points for point_type with new formatting
            pre['end_points'][synanno_type] =  this_type_annotations  
        
        # if the node has post-synaptic annotations (the current cell is assumed pre-synaptic)
        if pre['end_points'][synanno_type] != []:
            # for each synapse
            for syn_ in pre['end_points'][synanno_type]:
                '''assumes that the annotation is a point annotation stored in the list as ([x,y,z,segment_id],'annotatePoint')
                previous ot Jan 25 2024, it was just [x,y,z,segment_id]'''
                syn_ = syn_[0]
                try:
                    post_seg = syn_[3]
                    syn_ = array([int(syn_[i]) for i in range(3)]) # synapses annotations exported as nanometers, so do not need to convert

                    # go through each other nodes
                    for x_post in nodefiles.keys():
                        # if cell_type[x_post] in network_types:
                        post = base_segments[x_post] 
                        for k,v in post.items():
                            for v_ in list(v): #find keys (can be multiple on the same cell) for matching segment ids
                                if post_seg == v_: 
                                    # add edge to the graph between current node and matching node
                                    edge_list.append([x_pre,x_post,k,syn_[0],syn_[1],syn_[2]])
                                        

                except IndexError as msg:
                    cellid = x_pre
                    print(msg, f'for cell {cellid} synapse at {array([int(syn_[i]/vx_sizes[i]) for i in range(3)])} voxels has no segment id')

            else:
                continue


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5025/5025 [46:09<00:00,  1.81it/s]


# Synapses dataframe

In [83]:
df_syn = pd.DataFrame(edge_list,columns = ['pre','post','structure','x','y','z'])


# save df_syn

In [ ]:
savepath = dirpath.parent / 'data_processed_published'

df_syn.to_csv(savepath / 'df_postsyn.csv') # switch file name to df_presyn.csv if querying and building presynaptic inputs graph